In [8]:
import pandas as pd
import re

df1 = pd.read_csv("Data/dataset1.csv")
df2 = pd.read_csv("Data/dataset2.csv")
df3 = pd.read_csv("Data/dataset3.csv")

In [9]:
# -----------------------------
# 2. Clean column names
# -----------------------------
def clean_columns(df):
    df.columns = (
        df.columns
        .str.replace("\n", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

df1 = clean_columns(df1)
df2 = clean_columns(df2)
df3 = clean_columns(df3)

# # -----------------------------
# # 3. Standardize picture column
# # -----------------------------
# df1 = df1.rename(columns={
#     "Pictures": "Link to Picture",
#     "Link To Picture": "Link to Picture"
# })

# df2 = df2.rename(columns={
#     "Pictures": "Link to Picture",
#     "Link To Picture": "Link to Picture"
# })

# df3 = df3.rename(columns={
#     "Link": "Link to Picture",
#     "Link To Picture": "Link to Picture"
# })

# -----------------------------
# 4. Clean broken Group 3 DMS text
# -----------------------------
def clean_dms_text(value):
    if pd.isna(value):
        return value
    
    value = str(value).strip()

    replacements = {
        "Â°": "°",
        "â€™": "'",
        "’": "'",
        "â€": '"',
        "“": '"',
        "”": '"',
        "″": '"',
        "′": "'"
    }

    for bad, good in replacements.items():
        value = value.replace(bad, good)

    value = re.sub(r"\s+", " ", value).strip()
    return value

# -----------------------------
# 5. Convert DMS to decimal
# -----------------------------
def dms_to_decimal(value):
    if pd.isna(value):
        return None

    value = clean_dms_text(value)

    try:
        return float(value)
    except:
        pass

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*°\s*(\d+(?:\.\d+)?)\s*\'\s*(\d+(?:\.\d+)?)\s*"\s*([NSEW])',
        value
    )

    if not match:
        return None

    degrees, minutes, seconds, direction = match.groups()
    decimal = float(degrees) + float(minutes)/60 + float(seconds)/3600

    if direction in ["S", "W"]:
        decimal *= -1

    return decimal

df3["Latitude"] = df3["Latitude"].apply(clean_dms_text).apply(dms_to_decimal)
df3["Longitude"] = df3["Longitude"].apply(clean_dms_text).apply(dms_to_decimal)


# -----------------------------
# 7. Merge
# -----------------------------
combined_df = pd.concat([df1, df2, df3], ignore_index=True, sort=False)
combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

# -----------------------------
# 8. Fill missing values
# -----------------------------
combined_df = combined_df.fillna("N/A")

# -----------------------------
# 9. Save
# -----------------------------
combined_df.to_csv("data_combined.csv", index=False)
# -----------------------------
# 10. Preview
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

combined_df.head(10)

,Observation ID,Name,Data Type,Latitude,Longitude,Color,Material,Town,Link to Picture,Open/Closed,Number of People,Price Range (CHF)
0,G1_R0,Solar Panel 1,Solar Panel,45.90187,8.97062,Black,crystalline silicon,Riva San Vitale,https://drive.google.com/file/d/1P91eS5RCPLk_TuPHWM9bnzRaUt6PiUlj/view?usp=drive_link,N/A,N/A,N/A
1,G1_R1,Statues 1,Statues,45.9044,8.97055,Pale off-white/cream,lime-plastered masonry,Riva San Vitale,https://drive.google.com/file/d/1tgSBeQguIu10f5zDgBp02bva-xZ01Us-/view?usp=drive_link,N/A,N/A,N/A
2,G1_R2,Solar Panel 2,Solar Panel,45.90696,8.97095,Black,crystalline silicon,Riva San Vitale,https://drive.google.com/file/d/1kngJeMCJY5VlB6FlyZPtl58rvjKAhYR3/view?usp=sharing,N/A,N/A,N/A
3,G1_R3,Floors 1,Floors/Bridges,45.90608,8.97254,Gray,Pre-cast concrete (cement + sand + crushed aggregate with iron-oxide pigments,Riva San Vitale,https://drive.google.com/file/d/1y7Ue-Iqir3_Tj5Hbs8yPCeWHOEIM9I7I/view?usp=sharing,N/A,N/A,N/A
4,G1_R4,Floors 2,Floors/Bridges,45.90475,8.97052,Light Gray,natural igneous stone,Riva San Vitale,https://drive.google.com/file/d/1X3TihpIcc6KJDSuwy2_-2CIxiJfV8DLv/view?usp=sharing,N/A,N/A,N/A
5,G1_R5,Fountain 1,Fountains,45.90572,8.9713,Light Gray,granite,Riva San Vitale,https://drive.google.com/file/d/1D0GOe--hKk6VdRS2iYmQEIBkBfKWa7YV/view?usp=sharing,N/A,N/A,N/A
6,G1_R6,Fountain 2,Fountains,45.90338,8.97015,Gray,porous calcareous stone,Riva San Vitale,https://drive.google.com/file/d/1Gyy81htRjRq1Mv0H5KtIlnfnb4xLPglN/view?usp=drive_link,N/A,N/A,N/A
7,G1_R7,Statues 2,Statues,45.90538,8.9711,Green,bronze,Riva San Vitale,https://drive.google.com/file/d/1o0Vgj0eUTNCNaa0Rt9UY5M3_vbr_3-6z/view?usp=drive_link,N/A,N/A,N/A
8,G1_R8,Floors 3,Floors/Bridges,45.90533,8.97229,Light Gray,concrete,Riva San Vitale,https://drive.google.com/file/d/125CT2f_R9yzz8LjEHoVz6t_ModQXlZc3/view?usp=sharing,N/A,N/A,N/A
9,G1_R9,Floors 4,Floors/Bridges,45.90469,8.97089,Gray,limestone,Riva San Vitale,https://drive.google.com/file/d/1ZuanmT2az0urOML86uZm60cjH94AYoJZ/view?usp=sharing,N/A,N/A,N/A


In [10]:
import pandas as pd
import re

# -----------------------------
# 1. Load the new combined file
# -----------------------------
df = pd.read_csv("Data/combined_group_dataset_two.csv")

# -----------------------------
# 2. Clean column names
# -----------------------------
def clean_columns(df):
    df.columns = (
        df.columns
        .str.replace("\n", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

df = clean_columns(df)

# -----------------------------
# 3. Standardize Town names
# -----------------------------
def clean_town(value):
    if pd.isna(value):
        return value
    
    value = str(value).strip().lower()

    if value in ["riva", "riva san vitale"]:
        return "Riva San Vitale"
    
    return value.title()

if "Town" in df.columns:
    df["Town"] = df["Town"].apply(clean_town)

# -----------------------------
# 4. Standardize picture column names
# -----------------------------
df = df.rename(columns={
    "Pictures": "Link to Picture",
    "Link To Picture": "Link to Picture",
    "Link": "Link to Picture"
})

# -----------------------------
# 5. Clean broken DMS text
# -----------------------------
def clean_dms_text(value):
    if pd.isna(value):
        return value
    
    value = str(value).strip()

    replacements = {
        "Â°": "°",
        "â€™": "'",
        "’": "'",
        "â€": '"',
        "“": '"',
        "”": '"',
        "″": '"',
        "′": "'"
    }

    for bad, good in replacements.items():
        value = value.replace(bad, good)

    value = re.sub(r"\s+", " ", value).strip()
    return value

# -----------------------------
# 6. Convert DMS to decimal
# -----------------------------
def dms_to_decimal(value):
    if pd.isna(value):
        return None

    value = clean_dms_text(value)

    try:
        return float(value)
    except:
        pass

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*°\s*(\d+(?:\.\d+)?)\s*\'\s*(\d+(?:\.\d+)?)\s*"\s*([NSEW])',
        value
    )

    if not match:
        return value

    degrees, minutes, seconds, direction = match.groups()
    decimal = float(degrees) + float(minutes)/60 + float(seconds)/3600

    if direction in ["S", "W"]:
        decimal *= -1

    return decimal

if "Latitude" in df.columns:
    df["Latitude"] = df["Latitude"].apply(dms_to_decimal)

if "Longitude" in df.columns:
    df["Longitude"] = df["Longitude"].apply(dms_to_decimal)

# -----------------------------
# 7. Remove duplicate columns
# -----------------------------
df = df.loc[:, ~df.columns.duplicated()]

# -----------------------------
# 8. Fill missing values
# -----------------------------
df = df.fillna("N/A")

# -----------------------------
# 9. Save cleaned file
# -----------------------------
df.to_csv("combined_group_data_two_cleaned.csv", index=False)

# -----------------------------
# 10. Preview
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

df.head(10)

PermissionError: [Errno 13] Permission denied: 'combined_group_data_two_cleaned.csv'

In [23]:
import pandas as pd
import folium
from branca.element import Template, MacroElement


# load file
combined_df = pd.read_csv("combined_group_data_two_cleaned.csv")

# force coordinates to numeric
combined_df["Latitude"] = pd.to_numeric(combined_df["Latitude"], errors="coerce")
combined_df["Longitude"] = pd.to_numeric(combined_df["Longitude"], errors="coerce")

# drop rows with missing coordinates
map_df = combined_df.dropna(subset=["Latitude", "Longitude"]).copy()

# create map
m = folium.Map(location=[45.90, 8.97], zoom_start=14)
# m = folium.Map(location=[45.90, 8.97], zoom_start=14, tiles="CartoDB positron")


# color function
def get_color(data_type):
    data_type = str(data_type)
    if "Restaurant" in data_type:
        return "red"
    elif "Church" in data_type or "Cultural" in data_type:
        return "blue"
    elif "Retail" in data_type:
        return "purple"
    elif "Building" in data_type or "Solar" in data_type:
        return "green"
    else:
        return "gray"

# add markers
for _, row in map_df.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color=get_color(row["Data Type"]),
        fill=True,
        fill_opacity=0.7,
        popup=f"""
        <b>{row['Name']}</b><br>
        Type: {row['Data Type']}<br>
        Town: {row['Town']}
        """
    ).add_to(m)

legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed; 
    bottom: 50px; left: 50px; width: 220px; height: 160px; 
    background-color: white; 
    border:2px solid grey; 
    z-index:9999; 
    font-size:14px;
    padding: 10px;
    border-radius: 8px;
    ">
    
    <b>Legend</b><br><br>

    <i style="background:red; width:10px; height:10px; display:inline-block;"></i>
    Restaurant<br>

    <i style="background:blue; width:10px; height:10px; display:inline-block;"></i>
    Cultural / Church<br>

    <i style="background:purple; width:10px; height:10px; display:inline-block;"></i>
    Retail<br>

    <i style="background:green; width:10px; height:10px; display:inline-block;"></i>
    Building / Solar<br>

    <i style="background:gray; width:10px; height:10px; display:inline-block;"></i>
    Other
</div>
{% endmacro %}
"""

legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

# display
m


# conclusion 
# Cultural sites cluster --> historic center

In [22]:
m.save("town_map.html")